# Praktikum 6 Big Data - Apache Spark

Notebook ini memuat langkah praktikum, tugas, dan eksperimen lazy evaluation berdasarkan modul. Jalankan sel dari atas ke bawah di Google Colab.

## 1. Instalasi dan SparkSession

In [ ]:
!pip install pyspark -q

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, count as spark_count, avg as spark_avg

spark = (SparkSession.builder
         .appName("PraktikumBigData")
         .master("local[*]")
         .getOrCreate())

print("Versi Spark:", spark.version)
print("Spark Application Name:", spark.sparkContext.appName)
print("Spark Master:", spark.sparkContext.master)

## 2. E.1 - RDD

Hasil yang diharapkan: count 10; map menghasilkan kelipatan dua; filter mengambil angka > 50; gabungan filter genap dan map x3 menghasilkan `[60, 120, 180, 240, 300]`.

In [ ]:
data = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
rdd = spark.sparkContext.parallelize(data)

print("Jumlah elemen:", rdd.count())
print("Isi RDD:", rdd.collect())

# Transformation: mengalikan setiap elemen dengan 2
rdd_hasil = rdd.map(lambda x: x * 2)
print("Hasil map:", rdd_hasil.collect())

# Transformation: mengambil elemen yang lebih besar dari 50
rdd_filter = rdd.filter(lambda x: x > 50)
print("Hasil filter:", rdd_filter.collect())

# Mengambil bilangan genap, lalu mengalikannya dengan 3
rdd_genap = rdd.filter(lambda x: x % 2 == 0)
rdd_kali = rdd_genap.map(lambda x: x * 3)
print("Genap dikali 3:", rdd_kali.collect())

## 3. E.2 - Dataset dan DataFrame

Dataset transaksi disalin dari modul praktikum halaman 6.

In [ ]:
data_transaksi = [
    (1, "Laptop", "Elektronik", 7500000, 2),
    (2, "Mouse", "Aksesoris", 150000, 5),
    (3, "Keyboard", "Aksesoris", 300000, 3),
    (4, "Monitor", "Elektronik", 2500000, 2),
    (5, "Flashdisk", "Aksesoris", 80000, 10),
    (6, "Printer", "Elektronik", 1800000, 1),
    (7, "Headset", "Aksesoris", 250000, 4),
    (8, "Laptop", "Elektronik", 7500000, 1),
    (9, "Mouse", "Aksesoris", 150000, 8),
    (10, "Monitor", "Elektronik", 2500000, 3)
]

kolom = ["id_transaksi", "produk", "kategori", "harga", "jumlah"]
df = spark.createDataFrame(data_transaksi, kolom)
df_total = df.withColumn("total", col("harga") * col("jumlah"))

# Menampilkan data dan struktur

df.show()
df.printSchema()
print("Jumlah transaksi:", df.count())

# Memilih kolom tertentu

df.select("produk", "kategori", "harga").show()

# Filter harga lebih dari Rp1.000.000

df.filter(col("harga") > 1000000).show()

# Menambahkan kolom total transaksi dan mengurutkan terbesar ke terkecil

df_total.show()
df_total.orderBy(col("total").desc()).show()

## 4. E.3 - Agregasi DataFrame API

In [ ]:
# E.3 - Agregasi per kategori
hasil_agregasi = (df_total.groupBy("kategori")
    .agg(
        spark_sum("total").alias("total_pendapatan"),
        spark_count("*").alias("jumlah_transaksi"),
        spark_avg("total").alias("rata_rata_nilai_transaksi")
    ))
hasil_agregasi.show()

# Statistik keseluruhan
statistik_total = df_total.agg(
    spark_sum("total").alias("total_penjualan"),
    spark_avg("total").alias("rata_rata_penjualan"),
    spark_count("*").alias("jumlah_transaksi")
)
statistik_total.show()

# Tugas 2 - rata-rata jumlah produk terjual per kategori
rata_jumlah_kategori = (df_total.groupBy("kategori")
    .agg(
        spark_sum("total").alias("total_pendapatan"),
        spark_count("*").alias("jumlah_transaksi"),
        spark_avg("jumlah").alias("rata_rata_jumlah_produk")
    ))
rata_jumlah_kategori.show()

Hasil pemeriksaan manual: Elektronik Rp36.800.000 (5 transaksi), Aksesoris Rp4.650.000 (5 transaksi), total Rp41.450.000. Rata-rata unit per transaksi masing-masing kategori: 1,8 dan 6.

## 5. E.4 - Spark SQL

In [ ]:
# Membuat temporary view

df_total.createOrReplaceTempView("transaksi")

# Menampilkan seluruh data
spark.sql("SELECT * FROM transaksi").show()

# Filtering: total transaksi lebih dari Rp1.000.000
spark.sql("""
    SELECT produk, kategori, harga, jumlah, total
    FROM transaksi
    WHERE total > 1000000
""").show()

# Agregasi berdasarkan kategori
spark.sql("""
    SELECT kategori,
           SUM(total) AS total_pendapatan,
           COUNT(*) AS jumlah_transaksi,
           AVG(total) AS rata_rata_transaksi
    FROM transaksi
    GROUP BY kategori
""").show()

# Tugas 4: pendapatan per produk, urut terbesar ke terkecil
spark.sql("""
    SELECT produk, SUM(total) AS total_pendapatan
    FROM transaksi
    GROUP BY produk
    ORDER BY total_pendapatan DESC
""").show()

## 6. F - Lazy Evaluation

In [ ]:
# Transformation belum langsung menampilkan/menghasilkan hasil ke layar

df_filter = df.filter(col("harga") > 1000000)
print("Transformation selesai")

# Action memicu eksekusi dan menampilkan hasil

df_filter.show()

# Memeriksa rencana eksekusi

df_filter.explain()

## 7. Tugas Praktikum 6

In [ ]:
# Tugas 1 - Lima produk dengan total pendapatan tertinggi

(df_total.groupBy("produk")
    .agg(spark_sum("total").alias("total_pendapatan"))
    .orderBy(col("total_pendapatan").desc())
    .limit(5)
    .show())

# Tugas 2 - Total pendapatan, jumlah transaksi, rata-rata jumlah produk per kategori

(df_total.groupBy("kategori")
    .agg(
        spark_sum("total").alias("total_pendapatan"),
        spark_count("*").alias("jumlah_transaksi"),
        spark_avg("jumlah").alias("rata_rata_jumlah_produk")
    )
    .show())

# Tugas 3 - Harga > Rp100.000 dan jumlah minimal 3 unit

(df_total.filter((col("harga") > 100000) & (col("jumlah") >= 3))
    .select("id_transaksi", "produk", "kategori", "harga", "jumlah", "total")
    .show())

# Tugas 4 - Spark SQL

spark.sql("""
    SELECT produk, SUM(total) AS total_pendapatan
    FROM transaksi
    GROUP BY produk
    ORDER BY total_pendapatan DESC
""").show()

## 8. Jawaban konsep singkat

- **RDD**: kumpulan elemen terdistribusi tanpa schema tabel; cocok untuk pemrosesan elemen atau transformasi khusus. Contoh: `map()` dan `filter()` pada kumpulan angka.
- **DataFrame**: data terdistribusi berbentuk baris dan kolom dengan schema; cocok untuk data terstruktur. Contoh: mengelompokkan transaksi berdasarkan kategori lalu menghitung total pendapatan.
- **Transformation** bersifat lazy dan membentuk RDD/DataFrame baru. **Action** memicu eksekusi dan menghasilkan nilai/hasil.
- **Spark SQL** dan **DataFrame API** adalah dua cara mengolah data terstruktur di Spark; hasilnya sama jika dataset dan logika query sama.
- **collect()** mengirim semua elemen ke driver, sehingga sebaiknya dihindari untuk hasil yang sangat besar.